# 02a — CSV Data Exploration (sample)

**From Clinical Case Reports to Knowledge Graphs**

Explores the fixed 50-article CSV sample produced by
[`01_data_preparation.ipynb`](01_data_preparation.ipynb) in
`data/csv/sample/`. DuckDB queries the CSV files directly through
`read_csv_auto` — no separate database file is involved — so the same SQL
used here works unchanged against the full CSV export in
[`02b_csv_data_exploraton_full.ipynb`](02b_csv_data_exploraton_full.ipynb)
and the DuckDB database in
[`03_duckdb_data_exploraton.ipynb`](03_duckdb_data_exploraton.ipynb). Compare
results across the three to see what a 50-article sample distorts (e.g. the
journal/license breakdowns) versus what it preserves (e.g. text length
distribution).

Because array-typed columns (`authors`, `mesh_terms`, `major_mesh_terms`,
`keywords`) round-trip through CSV as bracketed text rather than native
arrays, `DESCRIBE` below shows them as `VARCHAR` — see Section 5 of
`01_data_preparation.ipynb` for why.


## 1. Setup


In [1]:
from pathlib import Path

import duckdb

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent
CSV_DIR = PROJECT_DIR / "data" / "csv" / "sample"

if not (CSV_DIR / "cases.csv").exists():
    raise FileNotFoundError(
        f"{CSV_DIR} not found — run 01_data_preparation.ipynb first."
    )

con = duckdb.connect()  # in-memory
for table in ("cases", "metadata", "data_dictionary"):
    con.execute(
        f"CREATE VIEW {table} AS SELECT * FROM read_csv_auto('{(CSV_DIR / f'{table}.csv').as_posix()}')"
    )

con.sql("SHOW TABLES")


┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ cases           │
│ data_dictionary │
│ metadata        │
└─────────────────┘

## 2. Example queries — exploring the sample


## Table schema


In [2]:
con.sql("DESCRIBE cases")


┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ article_id  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ age         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ case_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ case_text   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ gender      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [3]:
con.sql("DESCRIBE metadata")


┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ article_id       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ authors          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ case_amount      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ doi              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ journal          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ journal_detail   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keywords         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ license          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ link             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ major_mesh_terms │ VARC

### A random sample of cases


In [4]:
con.sql("SELECT * FROM cases LIMIT 5")


┌────────────┬────────┬───────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [5]:
con.sql("SELECT * FROM metadata LIMIT 5")


┌────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────┬──────────────────────────────────┬─────────────────────────┬───────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────┬─────────────┬───────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────┬──────────┬────────────────────────────────────────────────────────────────

### Text length distribution


In [6]:
con.sql("""
    SELECT
        MIN(LENGTH(case_text)) AS min_chars,
        MEDIAN(LENGTH(case_text)) AS median_chars,
        AVG(LENGTH(case_text))::INT AS avg_chars,
        MAX(LENGTH(case_text)) AS max_chars
    FROM cases
""")


┌───────────┬──────────────┬───────────┬───────────┐
│ min_chars │ median_chars │ avg_chars │ max_chars │
│   int64   │    double    │   int32   │   int64   │
├───────────┼──────────────┼───────────┼───────────┤
│       465 │       2884.0 │      3309 │     11806 │
└───────────┴──────────────┴───────────┴───────────┘

### Patient demographics


In [7]:
con.sql("""
    SELECT gender, COUNT(*) AS n_cases
    FROM cases
    GROUP BY gender
    ORDER BY n_cases DESC
""")


┌─────────┬─────────┐
│ gender  │ n_cases │
│ varchar │  int64  │
├─────────┼─────────┤
│ Female  │      30 │
│ Male    │      22 │
│ Unknown │       4 │
└─────────┴─────────┘

In [8]:
con.sql("""
    SELECT
        MIN(age) AS min_age,
        MEDIAN(age) AS median_age,
        AVG(age)::INT AS avg_age,
        MAX(age) AS max_age
    FROM cases
    WHERE age IS NOT NULL
""")


┌─────────┬────────────┬─────────┬─────────┐
│ min_age │ median_age │ avg_age │ max_age │
│ double  │   double   │  int32  │ double  │
├─────────┼────────────┼─────────┼─────────┤
│     0.0 │       40.0 │      38 │    83.0 │
└─────────┴────────────┴─────────┴─────────┘

### Joining `cases` with `metadata`

`cases` and `metadata` share the `article_id` (PMCID) column, so we can
bring in publication year, journal, license, etc. With only 50 articles sampled, expect the year/journal/license breakdowns below to be noisy or sparse compared to `02b_csv_data_exploraton_full.ipynb` — that's the point of comparing sample against full.


In [9]:
con.sql("""
    SELECT m.year, COUNT(*) AS n_cases
    FROM cases AS c
    JOIN metadata AS m USING (article_id)
    GROUP BY m.year
    ORDER BY m.year
""")


┌───────┬─────────┐
│ year  │ n_cases │
│ int64 │  int64  │
├───────┼─────────┤
│  1999 │       1 │
│  2008 │       1 │
│  2009 │       1 │
│  2013 │       2 │
│  2014 │       5 │
│  2015 │       2 │
│  2016 │       3 │
│  2017 │       1 │
│  2018 │       6 │
│  2019 │       3 │
│  2020 │       5 │
│  2021 │       1 │
│  2022 │       6 │
│  2023 │       7 │
│  2024 │       6 │
│  2025 │       5 │
│  2026 │       1 │
└───────┴─────────┘
      17 rows    

In [10]:
con.sql("""
    SELECT journal, COUNT(DISTINCT article_id) AS n_articles
    FROM metadata
    GROUP BY journal
    ORDER BY n_articles DESC
    LIMIT 10
""")


┌─────────────────────────────────────┬────────────┐
│               journal               │ n_articles │
│               varchar               │   int64    │
├─────────────────────────────────────┼────────────┤
│ Case Rep Crit Care                  │          2 │
│ Front Pediatr                       │          2 │
│ Surg Neurol Int                     │          2 │
│ J Investig Med High Impact Case Rep │          2 │
│ Ann Pediatr Cardiol                 │          2 │
│ Case Rep Obstet Gynecol             │          2 │
│ Surg Case Rep                       │          1 │
│ BMC Gastroenterol                   │          1 │
│ JA Clin Rep                         │          1 │
│ J Cardiovasc Echogr                 │          1 │
└─────────────────────────────────────┴────────────┘
  10 rows                                2 columns

In [11]:
con.sql("""
    SELECT license, COUNT(*) AS n_articles
    FROM metadata
    GROUP BY license
    ORDER BY n_articles DESC
""")


┌─────────────┬────────────┐
│   license   │ n_articles │
│   varchar   │   int64    │
├─────────────┼────────────┤
│ CC BY       │         31 │
│ CC BY-NC-SA │         11 │
│ CC BY-NC    │          8 │
└─────────────┴────────────┘

## Wrap up


In [12]:
con.close()
